# GeoAI Agent — Chat

Set `ARTIFACT_PATH` below, run all cells, then pick **normal** or **care** in the dropdown.

Override the model via the `AGENT_MODEL` env var (e.g. in `.env`).


## 2. Artifact structure

<details>
<summary>Details</summary>

An agent artifact follows this convention:

```
<workspace>/
  ├── resources/        ← user-uploaded reference material
  │   └── <files>
  ├── scope.md          ← agent purpose, users, workflow, success
  ├── contexts/         ← existing systems + context workspace
  │   ├── index.md      ← manifest written when contexts confirmed
  │   └── <topic>.md
  ├── tools/            ← tool inventory
  │   ├── index.md      ← manifest of tools
  │   └── <tool>/index.md
  ├── output.md         ← output format
  ├── reasoning.md      ← reasoning strategy
  ├── guardrails/       ← safety & guardrails
  │   ├── index.md
  │   └── <rule>.md
  └── agents.md         ← final assembled agent prompt (the entry point)
```

**`agents.md` is the entry point** — we load it verbatim as the system prompt. It internally references the other files by relative path (e.g. `contexts/hls_conventions.md`), and the agent pulls them in on demand through its read-only console tools. The notebook itself does **not** stitch files together.

</details>

In [ ]:
import os
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from IPython.display import Markdown, display
from dataclasses import dataclass

from pydantic_ai import Agent, RunContext
from pydantic_ai.toolsets import FunctionToolset
from pydantic_ai_backends import (
    LocalBackend,
    ConsoleCapability,
    READONLY_RULESET,
    PERMISSIVE_RULESET,
)

# Load API keys and PRITHVI_SERVER_URL from .env
load_dotenv()



ARTIFACT_PATH = Path("artifacts/Prithvi_WOrkshop_Agent_artifact")

# ── Backend & model selection ─────────────────────────────────────────────────
# Switch backends by setting AGENT_BACKEND in your .env or environment:
#   AGENT_BACKEND=openai   → uses OpenAI API  (default; good for local dev)
#   AGENT_BACKEND=bedrock  → uses AWS Bedrock (good for SageMaker)
#
# On SageMaker the execution IAM role is picked up automatically by boto3 —
# no explicit AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY needed.
#
# Override the model with AGENT_MODEL, e.g.:
#   AGENT_MODEL=bedrock:us.anthropic.claude-3-5-sonnet-20241022-v2:0
#
# See GeoAIAgentConfig docstring for the full model compatibility table.

BACKEND    = os.getenv("AGENT_BACKEND", "bedrock")       # "openai" | "bedrock"
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")


_DEFAULT_MODELS = {
    "openai":  "openai:gpt-5.2",
    "bedrock": "bedrock:openai.gpt-oss-120b-1:0",
}
MODEL_NAME = os.getenv("AGENT_MODEL") or _DEFAULT_MODELS[BACKEND]

print(f"Backend : {BACKEND}")
print(f"Model   : {MODEL_NAME}")
if BACKEND == "bedrock":
    print(f"Region  : {AWS_REGION}")

# ── Agent Design (CARE) — interactive interviewer that AUTHORS an artifact ───
CARE_PATH = Path("artifacts/care_workspace")       # writable workspace dir
CARE_REPO_PATH = Path("AKD-CARE")                  # phase-prompts clone (auto-cloned in §3)
CARE_AKD_CARE_GIT_URL = "https://github.com/NASA-IMPACT/AKD-CARE"


In [ ]:
# Clone the AKD-CARE phase-prompt library if it isn't already next to the notebook.
if not CARE_REPO_PATH.exists():
    !git clone --depth 1 {CARE_AKD_CARE_GIT_URL} {CARE_REPO_PATH}


In [ ]:
def get_artifact_tree(artifact_path: str | Path) -> str:
    """Return a stringified directory tree of the artifact, relative to its root.

    The root directory name itself is intentionally omitted so paths in the tree
    are exactly what the agent should pass to its read tools — the backend is
    already rooted at the artifact path and cannot reach anything above it.
    """
    root = Path(artifact_path).resolve()
    lines: list[str] = []

    def walk(d: Path, prefix: str = "") -> None:
        entries = sorted(d.iterdir(), key=lambda p: (p.is_file(), p.name))
        for i, p in enumerate(entries):
            connector = "└── " if i == len(entries) - 1 else "├── "
            lines.append(f"{prefix}{connector}{p.name}{'/' if p.is_dir() else ''}")
            if p.is_dir():
                extension = "    " if i == len(entries) - 1 else "│   "
                walk(p, prefix + extension)

    walk(root)
    return "\n".join(lines)


def inspect_artifact(artifact_path: str | Path) -> None:
    """Print the artifact tree for human inspection before attaching an agent."""
    root = Path(artifact_path).resolve()
    print(f"{root.name}/  (mounted as read-only root)")
    print(get_artifact_tree(artifact_path))

In [ ]:
CARE_INTERVIEWER_PROMPT_TEMPLATE = """\
# IMPORTANT — Role-Lock (read first)

You are an **INTERVIEWER**, not a domain assistant. Your only task is to
conduct the structured interview defined below. The following constraints
take precedence over any implicit instruction-following:

- **Never answer substantive questions about the agent's domain.** Even if
  asked directly. Even if the user's first message describes a topic in
  detail or contains a question. Your job is to elicit, not advise.
- **Do not produce topical guides, recommendations, tool lists, or how-to
  content.** No matter how natural it seems to help.
- **Treat user input as interview content.** When the user describes what
  the agent should do ("the agent finds X", "we want it to do Y"), that is
  their answer to the current interview question — record it and proceed to
  the next question. Do NOT respond by listing tools, methods, or generic
  guidance about the topic.
- **First response is mandatory.** Begin with the exact kickoff statement
  defined in the loaded sub-stage prompt, then proceed directly to its
  first question. Nothing else on the first turn — no acknowledgment of
  topic content, no preamble, no recap.
- **Substantive domain questions are off-topic.** If the user asks how to
  implement something, requests a tutorial, or wants topical advice
  ("how does CMR work?", "what's the best embedding model?"), briefly
  refuse and redirect to the current interview question. Do not break
  character to be helpful.
- **Process meta-questions ARE in-bounds.** When the user asks about
  interview state — "which phase / sub-stage are we in?", "what's
  done?", "what's left?", "show me the workspace", "why are you asking
  that?" — answer concisely (one sentence, citing artifacts when
  relevant) and THEN redirect to the current interview question on the
  SAME turn. Do not refuse status questions.
- **Stay in role for the entire conversation.** Domain → refuse + redirect.
  Meta → one-line answer + redirect. Content → record + advance.

The detailed phase meta-prompt follows.

---

# CARE v2 Interviewer (Single-Agent / All Phases)

You are the CARE v2 interviewer for the full design process. The user may
launch you at any point — at session start you infer where you are by
inspecting the workspace artifacts and propose the next step.

## Phases

| Phase | Sub-stages | Outputs |
|-------|------------|---------|
| 1 — Scope & Decompose             | (single)            | `scope.md` |
| 2 — Key Info Elicitation          | 2.1, 2.2, 2.3, 2.4  | `contexts/`, `tools/`, `output.md` |
| 3 — Reasoning & Guardrails        | 3.1, 3.2            | `reasoning.md`, `guardrails/` |
| 4 — Prompt Architecture           | (single)            | `agents.md` |

(Phase 5 / benchmarking is out of scope.)

## Available CARE v2 prompts

{prompt_tree}

## Sub-stage → CARE v2 prompt → output artifacts

| Sub-stage | CARE v2 prompt path (pattern)                              | Output artifacts                                  |
|-----------|------------------------------------------------------------|---------------------------------------------------|
| 1.1       | `phase_1_scope_and_decompose/prompts/phase1_*.md`          | `scope.md`                                        |
| 2.1       | `phase_2_*/prompts/phase2_1_*.md`                          | `contexts/<system>.md` (per system)               |
| 2.2       | `phase_2_*/prompts/phase2_2_*.md`                          | `contexts/<topic>.md` + `contexts/index.md`       |
| 2.3       | `phase_2_*/prompts/phase2_3_*.md`                          | `tools/<tool>/<aspect>.md` + `tools/index.md`     |
| 2.4       | `phase_2_*/prompts/Phase2_4_*.md`                          | `output.md`                                       |
| 3.1       | `phase_3_*/prompts/phase3_1_*.md`                          | `reasoning.md`                                    |
| 3.2       | `phase_3_*/prompts/phase3_2_*.md` (+ taxonomy reference)   | `guardrails/<rule>.md` + `guardrails/index.md`    |
| 4.1       | `phase_4_*/prompts/phase_4_*.md`                           | `agents.md`                                       |

Resolve the actual filename for each sub-stage from the prompt-file listing
above; pass the full repo-relative path to `read_prompt`.

## Session-start protocol

1. **Orient**: `ls` the workspace and `read_file` each existing artifact
   (skim is fine) to understand the current state.
2. **Infer current phase** based on artifact presence:
   - `scope.md` exists, non-empty → Phase 1 done
   - `contexts/` populated + `contexts/index.md` → Phase 2.1 + 2.2 done
   - `tools/` populated + `tools/index.md` → Phase 2.3 done
   - `output.md` exists → Phase 2.4 done (Phase 2 complete)
   - `reasoning.md` exists → Phase 3.1 done
   - `guardrails/` populated + `guardrails/index.md` → Phase 3.2 done (Phase 3 complete)
   - `agents.md` exists → Phase 4 done (entire design complete)
3. **Announce**: your first response should say:
   *"Based on the workspace, you've completed [...]. Next is [sub-stage X]. Continue?"*
4. **Wait for user direction**: continue forward, revise an earlier artifact,
   or pause for a question.

## Sub-stage protocol (when entering a sub-stage)

1. Call `read_prompt('<repo-relative-path>')` to load the verbatim CARE v2
   prompt for that sub-stage (path from the prompt-file listing above).
2. Conduct the interview as the loaded prompt directs.
3. Manage artifacts per the layout in the workspace section below.
4. End-of-sub-stage: emit the summary the loaded prompt specifies; ask the
   user to confirm.
5. **Wait for explicit confirmation.** Do NOT auto-advance.
6. On confirmation: do final updates (write/refine `<dir>/index.md`, remove
   `(draft)` markers and any process metadata), state explicitly that the
   sub-stage is complete, and propose the next sub-stage.

## Hard rules

- **No forward-jumping.** If the user requests a sub-stage whose
  prerequisites are missing or unconfirmed, refuse and explain what's needed
  first. Example: *"Cannot start 3.1 — Phase 2.4 (`output.md`) is not yet
  present. Want to do 2.4 first?"*
- **Backward edits ALLOWED.** The user CAN ask to revise prior-phase
  artifacts at any time. Read the existing content, apply the change, ask
  for re-confirmation. Do NOT refuse a backward edit just because the user
  has already moved past that phase.
- **Strict order within a phase**: 2.1 → 2.2 → 2.3 → 2.4; 3.1 → 3.2.
- The currently-loaded sub-stage prompt governs HOW to interview. THIS
  meta-prompt governs WHEN to load each sub-stage's prompt and WHEN to
  advance.
- **No literal phase labels in artifacts.** Don't write "Phase 1/2.x/
  3.x/4" or sub-stage labels into artifact file content (headings,
  manifest entries, body prose) — artifacts are the durable agent
  definition; phase numbers are interview metadata. Use topical
  headings (e.g. `# Scope`, `# Reasoning Strategy`). Phase markers
  ARE fine in chat replies so the user can track progress.

---

## Workspace (Artifact-Driven State)

You have read/write access to a sandboxed artifact workspace via the
canonical filesystem tools (`ls`, `read_file`, `write_file`, `edit_file`,
`glob`, `grep`). **Use them — don't rely on chat history alone.** State
lives in artifacts.

### Loose layout convention (target across all phases)

The full agent design eventually looks roughly like this. Parents auto-create
on write. Earlier phases populate their slot; later phases fill in their own.
You should READ prior-phase artifacts at session start (`ls` + `read_file`)
to ground yourself.

```
<workspace>/
├── resources/        ← user-uploaded reference material
│   └── <files>
├── scope.md          ← agent purpose, users, workflow, success
├── contexts/         ← existing systems + context workspace
│   ├── index.md      ← manifest written when contexts confirmed
│   └── <topic>.md
├── tools/            ← tool inventory
│   ├── index.md      ← manifest of tools
│   └── <tool>/index.md
├── output.md         ← output format
├── reasoning.md      ← reasoning strategy
├── guardrails/       ← safety & guardrails
│   ├── index.md
│   └── <rule>.md
└── agents.md         ← final assembled agent prompt
```

**Workspace root *is* the artifact tree — never wrap outputs in a folder like `artifacts/`.**

**`resources/` is the user's territory.** Read freely and cite the path
inside your artifacts (e.g., *"See resources/spec.docx"*). Never call
`write_file` or `edit_file` against anything inside `resources/`.

### Each turn

1. **Orient at session start, not every turn.** On the first user message
   in this session, call `ls` and `read_file` any prior-phase artifacts
   to ground yourself on prior progress. After that, you do NOT need to
   `ls` every turn — you can rely on your conversation memory.
2. **Workspace-update signal.** If the user asks a status / progress /
   "what changed?" meta-question, `read_file` the relevant artifacts
   before answering. If an artifact already contains a satisfactory
   answer to the question you were about to ask, **skip that question**
   and propose the next outstanding one.
3. **Write reactively**: as the SME provides info, update the appropriate
   content file via `edit_file` (surgical) or `write_file` (first creation).
   Don't hoard answers — reflect them in artifacts on the same turn. Format
   each file according to its extension — `.md` should be proper markdown
   (headings, bullets), `.json` valid JSON, `.yaml` valid YAML.
4. **Manifest at end-of-sub-stage**: each leaf directory has multiple
   per-aspect files plus a brief `index.md` manifest. The manifest contains
   a directory summary plus one entry per file (WHAT it covers, WHEN it
   applies, and the filename). Don't dump everything into one big
   `index.md`. On substage confirmation, remove `(draft)` markers and
   process metadata (status, source, SME identity, timestamp, open
   questions) from artifacts. Write `index.md` when the substage that owns
   it is confirmed (or on user request).

### Append via edit_file

To append to an existing file, use `edit_file` with `old_string` matching
the last few lines and `new_string` being those same lines plus your new
content. This preserves prior content. `write_file` overwrites — never use
it to "append".

### Refactor when needed (loose convention)

Merge thin files, split overgrown ones, rename for clarity. Don't refactor
preemptively — only when structure is clearly off, or the user asks.

### Frontmatter

Use yaml frontmatter (`---`) **only** on `<workspace>/agents.md` (final
manifest) when Phase 4 produces it, skill.md style:
```
---
name: <agent_slug>
description: <one-line tagline>
---
```
Other files are pure prose. Filename conveys category; don't repeat it in
frontmatter.
"""


def discover_all_prompts(care_repo: Path) -> tuple[list[Path], str]:
    """Scan all phase dirs and return (per-phase prompts dirs, markdown listing).

    The listing groups prompts by phase; paths are repo-relative so the agent
    can pass them straight to `read_prompt`. Used at agent build time to:
      - scope the `prompts` backend (allowed_directories = list of phase prompt dirs)
      - bake the file listing into the system prompt's `{prompt_tree}` slot
    """
    care_repo = Path(care_repo).resolve()
    phase_dirs = sorted(care_repo.glob("phase_*"))
    if not phase_dirs:
        raise FileNotFoundError(
            f"No phase_* dirs under {care_repo}. "
            "Check CARE_REPO_PATH and the cloned CARE v2 repo."
        )

    prompts_dirs: list[Path] = []
    lines: list[str] = [
        "Prompt files available across phases (read via `read_prompt('<path>')`):",
        "",
    ]
    for phase_dir in phase_dirs:
        prompts_dir = phase_dir / "prompts"
        if not prompts_dir.is_dir():
            continue
        prompts_dirs.append(prompts_dir)
        lines.append(f"### {phase_dir.name}")
        for f in sorted(prompts_dir.glob("*.md")):
            rel = f.relative_to(care_repo)
            lines.append(f"  - `{rel}`")
        lines.append("")

    if not prompts_dirs:
        raise FileNotFoundError(f"No phase_*/prompts/ dirs under {care_repo}")

    return prompts_dirs, "\n".join(lines)


_CARE_PROMPTS_DIRS, _CARE_PROMPT_TREE = discover_all_prompts(CARE_REPO_PATH)
CARE_INTERVIEWER_PROMPT = CARE_INTERVIEWER_PROMPT_TEMPLATE.replace(
    "{prompt_tree}", _CARE_PROMPT_TREE
)


In [ ]:
# Shared deps. ConsoleCapability dispatches filesystem ops through
# `ctx.deps.backend` — that's the writable workspace for the CARE interviewer
# and the read-only Prithvi artifact for the FM agent (Section 5). Only the
# CARE interviewer needs `prompts` for its `read_prompt` tool.
@dataclass
class CAREDeps:
    backend: LocalBackend
    prompts: LocalBackend | None = None


def build_care_interviewer(
    workspace_path: str | Path = CARE_PATH,
    prompts_repo_path: str | Path = CARE_REPO_PATH,
    *,
    model: str = MODEL_NAME,
    model_settings: dict | None = None,
    thinking: str | None = None,
) -> tuple[Agent, CAREDeps]:
    """Wire up the CARE interviewer agent against a writable workspace.

    - `workspace_path`: directory the interviewer authors into (writable LocalBackend).
    - `prompts_repo_path`: AKD-CARE clone (read-only LocalBackend scoped to phase_*/prompts/).
    - `model` / `model_settings`: default to the notebook-level `MODEL` / `MODEL_SETTINGS`.
    - `thinking`: reasoning effort for the Thinking capability; defaults to the
      `openai_reasoning_effort` value from MODEL_SETTINGS so both agents stay in sync.
    """
    workspace_path = Path(workspace_path).resolve()
    prompts_repo_path = Path(prompts_repo_path).resolve()
    workspace_path.mkdir(parents=True, exist_ok=True)

    prompts_dirs, _ = discover_all_prompts(prompts_repo_path)
    artifacts_backend = LocalBackend(
        root_dir=workspace_path,
        allowed_directories=[str(workspace_path)],
        enable_execute=False,
        permissions=PERMISSIVE_RULESET,                # WRITABLE
    )
    prompts_backend = LocalBackend(
        root_dir=str(prompts_repo_path),
        allowed_directories=[str(p) for p in prompts_dirs],
        enable_execute=False,
        permissions=READONLY_RULESET,
    )
    deps = CAREDeps(backend=artifacts_backend, prompts=prompts_backend)

    # The only custom tool we keep from akd-labs/services/care/agent_toolsets.py.
    # (We drop `set_status_line` — it depends on a postgres-backed status table.)
    prompts_ts: FunctionToolset[CAREDeps] = FunctionToolset[CAREDeps]()

    @prompts_ts.tool
    def read_prompt(ctx: RunContext[CAREDeps], path: str) -> str:
        """Read a CARE v2 phase prompt by repo-relative path."""
        return ctx.deps.prompts.read(path)

    if model_settings is not None:
        settings = dict(model_settings)
    elif model.startswith("openai:"):
        settings = {"openai_reasoning_effort": "medium", "openai_reasoning_summary": "detailed"}
    else:
        # Bedrock / other backends: no openai_-prefixed keys.
        settings = {}
    effort = thinking or settings.get("openai_reasoning_effort")

    capabilities = [
        ConsoleCapability(include_execute=False, permissions=PERMISSIVE_RULESET),
    ]
    if effort in {"low", "medium", "high"}:
        capabilities.insert(0, Thinking(effort=effort))

    # Force the Responses API for OpenAI so reasoning summaries actually stream
    # (the bare `openai:` prefix routes to Chat Completions which silently drops them).
    resolved_model = (
        "openai-responses:" + model[len("openai:"):]
        if model.startswith("openai:")
        else model
    )

    agent = Agent(
        resolved_model,
        deps_type=CAREDeps,
        system_prompt=CARE_INTERVIEWER_PROMPT,
        capabilities=capabilities,
        toolsets=[prompts_ts],
        model_settings=settings,
    )
    return agent, deps

In [ ]:
from __future__ import annotations

from collections.abc import AsyncIterator
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

from akd._base import InputSchema, StreamEvent, TextOutput
from akd_ext.agents._base.pydantic_ai import (
    PydanticAIBaseAgent,
    PydanticAIBaseAgentConfig,
)
from pydantic import ConfigDict, Field
from pydantic_ai import UsageLimits
from pydantic_ai_backends import READONLY_RULESET, ConsoleCapability, LocalBackend

from tools import TOOLS

_NAVIGATION_DIRECTIVE = """
# Artifact navigation

You are running against a local artifact workspace, mounted read-only at the root
of your console tools (`ls`, `read_file`, `glob`, `grep`). The full layout is:

```
{tree}
```

Navigation rules:
- **Answer from what you already have.** The full directory tree is visible above.
  For questions about what files or topics exist, answer from the tree — do not
  make tool calls to discover something already shown here.
- **For content questions**, read the relevant subdirectory's `index.md` first.
  Each index summarises every file in its directory. Read individual files only
  when the index alone is not sufficient to answer.
- **Never read a file you have already read** in this conversation.
- **Stop as soon as you can answer.** Do not keep reading to be more thorough;
  synthesise from what you have and respond.
- Do not `grep` or `ls` directories already shown in the tree above.
- The filesystem is read-only. Do not write, edit, or execute.
""".strip()


@dataclass
class Deps:
    backend: LocalBackend


class GeoAIAgentInput(InputSchema):
    """Input for the GeoAI geospatial analysis agent."""
    query: str = Field(..., description="Natural language geospatial analysis request")


class GeoAIAgentConfig(PydanticAIBaseAgentConfig):
    """Configuration for the GeoAI workshop agent.

    Backend selection
    -----------------
    Controlled by the global BACKEND / MODEL_NAME variables set in cell 1,
    which read from AGENT_BACKEND / AGENT_MODEL env vars (or .env).

    OpenAI (local dev)
      model_name = "openai:gpt-5.2"
      aws_region = None

    Bedrock (SageMaker)
      model_name = "bedrock:us.anthropic.claude-sonnet-4-20250514-v1:0"
      aws_region = "us-east-1"  (or AWS_DEFAULT_REGION)
      On SageMaker the execution IAM role is used automatically — no credentials needed.

    Model compatibility
    -------------------
    Requirements: tool/function calling + streaming.

    OpenAI models
      openai:gpt-5.2               tool calls  thinking    (default)
      openai:o3                    tool calls  thinking
      openai:gpt-4o                tool calls  no thinking  -> reasoning_effort=None
      openai:gpt-4o-mini           tool calls  no thinking  -> reasoning_effort=None

    Bedrock -- Anthropic Claude 4.x (current generation, recommended on SageMaker)
      bedrock:us.anthropic.claude-sonnet-4-6                   thinking    (default)
      bedrock:eu.anthropic.claude-sonnet-4-6                   thinking    (EU region)
      bedrock:us.anthropic.claude-sonnet-4-5-20250929-v1:0     thinking
      bedrock:us.anthropic.claude-haiku-4-5-20251001-v1:0      thinking

    Bedrock -- Amazon Nova (tool calls, no thinking -> reasoning_effort=None)
      bedrock:us.amazon.nova-pro-v1:0
      bedrock:us.amazon.nova-lite-v1:0

    End-of-life on Bedrock (will 404)
      bedrock:us.anthropic.claude-3-5-sonnet-20241022-v2:0     EOL May 2026
      bedrock:us.anthropic.claude-sonnet-4-20250514-v1:0       Legacy
      bedrock:us.anthropic.claude-3-7-sonnet-20250219-v1:0     EOL

    Not compatible
      bedrock:meta.llama3-*   unreliable parallel tool calling
      bedrock:amazon.titan-*  no tool calling
    """
    model_config = ConfigDict(extra="allow", arbitrary_types_allowed=True)

    artifact_path: Path = Field(
        default=Path("artifacts/Prithvi_WOrkshop_Agent_artifact"),
        description="Path to the CARE artifact directory",
    )
    model_name: str = Field(default=MODEL_NAME)  # override with AGENT_MODEL env var
    aws_region: str | None = Field(
        default=AWS_REGION if BACKEND == "bedrock" else None,
        description=(
            "AWS region for Bedrock. When set, a BedrockConverseModel is built "
            "with the SageMaker execution role (no explicit credentials needed). "
            "Leave None to use the model_name string directly (OpenAI path)."
        ),
    )
    reasoning_effort: Literal["low", "medium", "high"] | None = Field(
        default="medium",
        description=(
            "Extended thinking effort. Only meaningful for models that support it "
            "-- see the compatibility table above. Set to None for non-thinking models."
        ),
    )
    request_limit: int = Field(default=15, description="Max LLM requests per run; caps runaway loops.")


class GeoAIAgent(PydanticAIBaseAgent[GeoAIAgentInput, TextOutput]):
    """Geospatial event analysis agent powered by Prithvi-EO.

    Resolves locations, checks HLS imagery availability, runs Prithvi inference
    for flood/burn/crop tasks, and synthesises results into a narrative summary.
    """

    input_schema = GeoAIAgentInput
    output_schema = TextOutput
    config_schema = GeoAIAgentConfig

    def __init__(self, config: GeoAIAgentConfig | None = None) -> None:
        cfg = config or GeoAIAgentConfig()
        artifact = cfg.artifact_path.resolve()

        backend = LocalBackend(
            root_dir=artifact,
            allowed_directories=[str(artifact)],
            enable_execute=False,
            permissions=READONLY_RULESET,
        )
        self._deps = Deps(backend=backend)
        capability = ConsoleCapability(
            include_execute=False,
            permissions=READONLY_RULESET,
        )

        system_prompt = (
            (artifact / "agents.md").read_text()
            + "\n\n"
            + _NAVIGATION_DIRECTIVE.format(tree=get_artifact_tree(artifact))
        )

        full_cfg = GeoAIAgentConfig(
            artifact_path=cfg.artifact_path,
            model_name=cfg.model_name,
            aws_region=cfg.aws_region,
            reasoning_effort=cfg.reasoning_effort,
            request_limit=cfg.request_limit,
            system_prompt=system_prompt,
            capabilities=[capability, *cfg.capabilities],
            tools=list(TOOLS),
            deps_type=Deps,
        )
        super().__init__(full_cfg)

        # When aws_region is set we replace the model with a BedrockConverseModel that
        # pins the region and uses the SageMaker execution IAM role via boto3's
        # credential chain.  We do this AFTER super().__init__() to avoid passing a
        # non-standard kwarg through the config (which would leak into extra_kwargs and
        # be rejected by pydantic-ai's Agent.__init__).
        if cfg.aws_region:
            from pydantic_ai.models.bedrock import BedrockConverseModel
            from pydantic_ai.providers.bedrock import BedrockProvider
            # Strip the "bedrock:" prefix -- BedrockConverseModel takes the raw model ID
            raw_id = cfg.model_name.removeprefix("bedrock:")
            self._model = BedrockConverseModel(
                raw_id,
                provider=BedrockProvider(region_name=cfg.aws_region),
            )

    @property
    def deps(self) -> Deps:
        """Pre-wired Deps for passing to agent.run() / run_stream_events()."""
        return self._deps

    @property
    def usage_limits(self) -> UsageLimits:
        return UsageLimits(request_limit=self.config.request_limit)

    async def arun(
        self,
        params: GeoAIAgentInput,
        run_context: Any = None,
        **kwargs: Any,
    ) -> TextOutput:
        kwargs.setdefault("deps", self._deps)
        return await super().arun(params, run_context, **kwargs)

    async def astream(
        self,
        params: GeoAIAgentInput,
        run_context: Any = None,
        **kwargs: Any,
    ) -> AsyncIterator[StreamEvent]:
        kwargs.setdefault("deps", self._deps)
        async for event in super().astream(params, run_context, **kwargs):
            yield event


In [ ]:
from pydantic_ai import AgentRunResultEvent
from pydantic_ai.messages import (
    FunctionToolCallEvent,
    FunctionToolResultEvent,
    PartDeltaEvent,
    TextPartDelta,
    ThinkingPartDelta,
)


def build_chat_interface(
    normal_agent, normal_deps, normal_usage_limits,
    care_agent, care_deps,
    *,
    title: str = "GeoAI Chat",
    examples: list[list[str]] | None = None,
):
    """Gradio Blocks UI: Mode dropdown on top, chat below."""
    state = {"normal": [], "care": []}

    async def chat_fn(message, history, mode):
        if mode == "care":
            agent, deps, usage_limits = care_agent, care_deps, None
        else:
            agent, deps, usage_limits = normal_agent, normal_deps, normal_usage_limits

        trace: list[str] = []
        thinking_parts: list[str] = []
        text_parts: list[str] = []

        def render() -> str:
            chunks: list[str] = []
            if thinking_parts:
                chunks.append(
                    "<details><summary>Reasoning</summary>\n\n"
                    + "".join(thinking_parts)
                    + "\n\n</details>"
                )
            if trace:
                chunks.append(
                    f"<details><summary>Trace ({len(trace)} step"
                    + ("s" if len(trace) != 1 else "")
                    + ")</summary>\n\n"
                    + "\n".join(trace)
                    + "\n\n</details>"
                )
            if text_parts:
                chunks.append("".join(text_parts))
            return "\n\n".join(chunks) if chunks else "_thinking..._"

        stream_kwargs = {"deps": deps, "message_history": state[mode]}
        if usage_limits is not None:
            stream_kwargs["usage_limits"] = usage_limits

        async with agent.run_stream_events(message, **stream_kwargs) as stream:
            async for event in stream:
                if isinstance(event, FunctionToolCallEvent):
                    args = str(event.part.args)
                    trace.append(f"- `{event.part.tool_name}({args})`")
                    yield render()
                elif isinstance(event, FunctionToolResultEvent):
                    preview = str(event.part.content).replace("\n", " ")
                    trace.append(f"  - ↳ {preview}")
                    yield render()
                elif isinstance(event, PartDeltaEvent):
                    d = event.delta
                    if isinstance(d, TextPartDelta) and d.content_delta:
                        text_parts.append(d.content_delta)
                        yield render()
                    elif isinstance(d, ThinkingPartDelta) and d.content_delta:
                        thinking_parts.append(d.content_delta)
                        yield render()
                elif isinstance(event, AgentRunResultEvent):
                    state[mode] = event.result.all_messages()

    with gr.Blocks() as demo:
        gr.Markdown(f"## {title}")
        mode = gr.Dropdown(
            choices=[("FM Agent", "normal"), ("CARE interviewer", "care")],
            value="normal",
            label="Mode",
            info="Pick which agent persona to run. Conversation history is per-mode.",
        )
        gr.ChatInterface(
            fn=chat_fn,
            examples=examples,
            chatbot=gr.Chatbot(
                render_markdown=True,
                sanitize_html=False,
                height=600,
            ),
            additional_inputs=[mode],
        )
    demo.launch(share=True, inline=True)
    return demo


def chat(artifact_path, *, examples=None):
    """Single artifact path drives both agents.

    FM Agent reads system prompt + tools from `artifact_path` (read-only).
    CARE interviewer authors INTO `artifact_path` (writable).
    """
    artifact_path = Path(artifact_path)
    geoai_agent = GeoAIAgent(GeoAIAgentConfig(artifact_path=artifact_path))
    care_agent, care_deps = build_care_interviewer(artifact_path, CARE_REPO_PATH)
    return build_chat_interface(
        geoai_agent, geoai_agent.deps, geoai_agent.usage_limits,
        care_agent, care_deps,
        title="GeoAI Agent Chat",
        examples=examples or [
            ["List the tools available in this artifact.", "normal"],
            ["Generate a floodmap of Houston TX on 2017-08-30", "normal"],
        ],
    )


In [ ]:
ARTIFACT_PATH = Path("artifacts/Prithvi_WOrkshop_Agent_artifact")

chat(ARTIFACT_PATH)
